[문서 형식 확장] docx, hwpx, pdf  +  md, txt, pptx, html, xlsx, …

In [32]:
# [역할] 실습 환경 준비: 프로젝트 루트로 이동하고, 재료·모델 경로를 정하고, backend를 import 경로에 추가
import os        # 작업 폴더 변경용
import pathlib   # 경로를 객체로 다루기
import sys       # 모듈 검색 경로(sys.path) 조작용

here = pathlib.Path.cwd()                                                  # 현재 작업 폴더
# ROOT = next(p for p in (here, *here.parents) if (p / 'backend').is_dir())  # 위로 올라가며 backend 폴더가 있는 곳 = 프로젝트 루트
ROOT = here.parents[2] if here.name == "Thu" else here      # Thu에서 실행했으면 3단계 위가 프로젝트 루트
os.chdir(ROOT)                                              # 작업 폴더를 루트로 이동
SANDBOX = ROOT / "sandbox" / "W5" / "Thu"                   # 오늘 실습 폴더
SAMPLES = ROOT / "sandbox" / "W5" / "Mon" / "samples"       # 샘플 문서(재료) 폴더
MODELS = ROOT.parent / 'models'                             # 임베딩 모델 폴더 (프로젝트 옆)
FORMATS = SAMPLES / '_formats'                              # 여러 형식 샘플 폴더

print("프로젝트 루트  :", ROOT)       # 루트 경로 확인
print("재료 폴더      :", SAMPLES)    # 샘플 폴더 경로 확인

if str(ROOT / 'backend') not in sys.path:          # backend가 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))      # 맨 앞에 추가 → `from app...` import 가능

프로젝트 루트  : d:\hanwha-agent
재료 폴더      : d:\hanwha-agent\sandbox\W5\Mon\samples


In [33]:
# [역할] 재료 확인: 샘플 폴더가 있는지, 파일이 몇 개인지, _formats 안에 어떤 형식이 있는지 출력
if not SAMPLES.is_dir():                                     # 샘플 폴더가 없으면
    print('재료 폴더가 없습니다:', SAMPLES)                     # 경로 출력
    print('9/17 노트북 00 번의 복사 셀을 한 번 더 돌리세요.')     # 해결 방법 안내
else:                                                        # 폴더가 있으면
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]     # 하위 폴더까지 모든 파일
    root_files = [p for p in SAMPLES.iterdir() if p.is_file()]     # samples 바로 아래 파일만
    fmt_files = [p for p in FORMATS.iterdir() if p.is_file()]      # _formats 폴더 안 파일
    print(f'재료 파일 : {len(all_files)}개')                                                  # 전체 파일 수
    print(f'  samples 바로 아래 : {len(root_files)}개 · _formats 안 : {len(fmt_files)}개')    # 위치별 파일 수
    print('_formats 안의 재료 :')                                  # 목록 제목
    for p in sorted(FORMATS.glob('sample_*')):                     # sample_로 시작하는 파일을 이름순으로
        print('  ' + p.name + ('   ← 오후 03 번' if p.suffix == '.pdf' else ''))   # pdf는 오후 실습용 표시


재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
_formats 안의 재료 :
  sample_guide.md
  sample_memo.txt
  sample_notice.pptx
  sample_page.html
  sample_scanned.pdf   ← 오후 03 번
  sample_table.xlsx


다른 형식의 파일이 현재 local_parser에서 파싱이 되는지 확인

In [34]:
# [역할] 문제 확인: 새 형식(xlsx/pptx/txt/md/html)을 기존 parse_local에 넣으면 모두 거절되는 것을 확인
from app.core.exceptions import ValidationFailed     # 입력 검증 실패 예외 (지원 안 하는 형식 등)
from app.rag.local_parsers import parse_local         # 현재 로컬 파서 (docx/hwpx/pdf만 지원)
today = ['sample_table.xlsx', 'sample_notice.pptx', 'sample_memo.txt', 'sample_guide.md', 'sample_page.html']   # 오늘 시험할 새 형식들
for name in today:                                    # 파일마다
    p = FORMATS / name                                # 전체 경로
    try:                                              # 실패해도 다음 파일로 넘어가도록
        doc = parse_local(p)                          # 파싱 시도
        if doc is None:                               # 파서 내부 오류로 None이 오면
            print(f'{p.name:<20}-> 파서가 도중에 터졌습니다 (로그를 보세요)')   # 실패 안내
        else:                                         # 성공하면
            print(f'{p.name:<20}-> 블록 {len(doc.blocks)}개')   # 블록 수 출력
    except ValidationFailed as e:                     # 지원하지 않는 형식이면
        print(f'{p.name:<20}-> ValidationFailed: {e.message}  | detail={e.detail}')   # 오류 메시지와 상세 출력

sample_table.xlsx   -> ValidationFailed: 지원하지 않는 형식입니다: .xlsx  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_notice.pptx  -> ValidationFailed: 지원하지 않는 형식입니다: .pptx  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_memo.txt     -> ValidationFailed: 지원하지 않는 형식입니다: .txt  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_guide.md     -> ValidationFailed: 지원하지 않는 형식입니다: .md  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_page.html    -> ValidationFailed: 지원하지 않는 형식입니다: .html  | detail=핸들러를 추가하면 지원할 수 있습니다


In [35]:
# [역할] txt 파일은 파이썬으로 바로 읽히는데, 파서는 확장자만 보고 거절한다는 것을 비교
from app.core.exceptions import ValidationFailed     # 지원 안 하는 형식일 때 나는 예외
from app.rag.local_parsers import parse_local         # 로컬 파서

memo = FORMATS / 'sample_memo.txt'                    # 텍스트 파일 하나
text = memo.read_text(encoding='utf-8')               # 파이썬으로 그냥 읽기
print(f'[그냥 읽으면]  {len(text)}자')                  # 글자 수
print(text)                                           # 내용 출력 → 읽는 건 아무 문제 없음
print('[parse_local 에 넣으면]')                       # 비교 제목
try:                                                  # 오류를 잡아서 보여 주기 위해
    parse_local(memo)                                 # 같은 파일을 파서에 넣으면
except ValidationFailed as e:                         # 확장자 때문에 거절됨
    print(f'ValidationFailed: {e} | detail={e.detail}')   # 오류 내용 출력

[그냥 읽으면]  71자
출장 정산 안내

9월부터 출장 정산은 그룹웨어에서만 접수합니다.
종이 서류는 받지 않습니다.

문의: 인사총무팀 김지원 과장

[parse_local 에 넣으면]
ValidationFailed: 지원하지 않는 형식입니다: .txt | detail=핸들러를 추가하면 지원할 수 있습니다


In [36]:
# [역할] 분기 설계 미리보기: 확장자에 따라 어느 파서로 보낼지 결정하는 규칙(which)을 만들고, 업스테이지 사용 여부에 따른 차이를 확인
_PLAIN = {'.txt', '.md'}          # 그냥 텍스트로 읽으면 되는 형식
_HTML = {'.html', '.htm'}         # HTML 형식
_LOCAL_ONLY = {'.hwp', '.hwpx'}   # 한글 파일은 업스테이지가 못 읽어서 항상 로컬 파서로

# 파일 이름 → 어느 파서로 보낼지 결정 (실제 분기 로직의 미리보기)
def which(name, use_upstage=False):                  # name: 파일 이름 · use_upstage: 업스테이지 사용 여부
    ext = pathlib.Path(name).suffix.lower()          # 확장자 (소문자로)
    if ext in _PLAIN:                                # txt, md면
        return '_parse_plain'                        # 텍스트 파서
    if ext in _HTML:                                 # html이면
        return '_parse_html'                         # HTML 파서
    if use_upstage and ext not in _LOCAL_ONLY:       # 업스테이지를 쓰기로 했고 한글 파일이 아니면
        return 'upstage'                             # 업스테이지 API
    return 'parse_local'                             # 나머지는 기존 로컬 파서
names = ['sample_memo.txt', 'sample_guide.md', 'sample_page.html', 'sample_table.xlsx', 'sample_notice.pptx', 'DOC-HR-014_국내출장_여비_규정_v2.0.docx', 'DOC-HR-014_국내출장_여비_규정_v2.0.hwpx']   # 시험할 파일 이름들
print('[use_upstage=False]')                         # 업스테이지 끈 경우 제목
for n in names:                                      # 업스테이지 없이
    print(f'  {n:<40} -> {which(n)}')                # 어느 파서로 가는지 출력
print()                                              # 빈 줄
print('[use_upstage=True]')                          # 업스테이지 켠 경우 제목
for n in names:                                      # 업스테이지를 켜면
    before, after = (which(n), which(n, use_upstage=True))   # 켜기 전/후 결과
    mark = '(바뀐다)' if before != after else '(그대로)'       # 달라졌는지 표시
    print(f'  {n:<40} -> {after:<14} {mark}')        # 켠 뒤 파서와 변화 여부 출력

[use_upstage=False]
  sample_memo.txt                          -> _parse_plain
  sample_guide.md                          -> _parse_plain
  sample_page.html                         -> _parse_html
  sample_table.xlsx                        -> parse_local
  sample_notice.pptx                       -> parse_local
  DOC-HR-014_국내출장_여비_규정_v2.0.docx          -> parse_local
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx          -> parse_local

[use_upstage=True]
  sample_memo.txt                          -> _parse_plain   (그대로)
  sample_guide.md                          -> _parse_plain   (그대로)
  sample_page.html                         -> _parse_html    (그대로)
  sample_table.xlsx                        -> upstage        (바뀐다)
  sample_notice.pptx                       -> upstage        (바뀐다)
  DOC-HR-014_국내출장_여비_규정_v2.0.docx          -> upstage        (바뀐다)
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx          -> parse_local    (그대로)


In [37]:
# [역할] txt 파싱 연습: 빈 줄(\n\n)을 기준으로 단락을 나눠 블록으로 만들기
memo=FORMATS/'sample_memo.txt'                                  # 텍스트 파일 경로
text=memo.read_text(encoding='utf-8',errors='replace')          # 읽기 (깨진 글자는 대체 문자로)
parts=[s.strip() for s in text.split('\n\n') if s.strip()]      # 빈 줄로 나누고, 앞뒤 공백 제거, 빈 단락 제외

for i,s in enumerate(parts, 1):                                 # 1번부터 번호 붙여서
    print(f'[{i}]{len(s)}자: {s}')                               # 번호, 글자 수, 내용 출력

print(f'블록 {len(parts)}개')                                    # 단락(블록) 수

[1]8자: 출장 정산 안내
[2]42자: 9월부터 출장 정산은 그룹웨어에서만 접수합니다.
종이 서류는 받지 않습니다.
[3]16자: 문의: 인사총무팀 김지원 과장
블록 3개


In [38]:
# [역할] md 파싱 연습: txt와 같은 방식(빈 줄 기준)으로 나누면 제목(#)과 본문이 각각 블록이 되는지 확인
guide=FORMATS / 'sample_guide.md'                               # 마크다운 파일 경로
text=guide.read_text(encoding='utf-8', errors='replace')        # 읽기 (깨진 글자는 대체 문자로)
parts=[s.strip() for s in text.split('\n\n') if s.split()]      # 빈 줄로 나누고, 내용 없는 단락 제외

for i,s in enumerate(parts,1):                                  # 1번부터 번호 붙여서
    print(f'[{i}]{len(s)}자:{s}')                                # 번호, 글자 수, 내용 출력

print(f'블록 {len(parts)}개')                                    # 블록 수

[1]13자:# 출장 신청 빠른 안내
[2]8자:## 1. 신청
[3]23자:출장 3일 전까지 그룹웨어에서 신청합니다.
[4]8자:## 2. 승인
[5]39자:팀장 승인 후 확정됩니다. 5일 초과 출장은 본부장 승인이 필요합니다.
[6]8자:## 3. 정산
[7]27자:복귀 후 7일 이내에 영수증을 첨부해 정산합니다.
블록 7개


In [39]:
# [역할] HTML 나쁜 예: 태그를 한꺼번에 지우면 본문과 표가 한 줄로 뭉개져 블록이 1개만 나오는 것을 확인
# bad example
import re                                                       # 정규식

page=FORMATS / 'sample_page.html'                               # HTML 파일 경로
html=page.read_text(encoding='utf-8', errors='replace')         # 원문 읽기

# 패턴으로 태그 형태 변경
flat=re.sub(r'\s+',' ',re.sub(r'<[^>]+>',' ',html)).strip()     # 모든 태그를 공백으로 → 연속 공백을 하나로

print(flat)                                                     # 결과: 줄바꿈 없이 한 줄

blocks=[s.strip() for s in flat.split('\n\n') if s.strip()]     # 빈 줄 기준으로 나누려 해도 줄바꿈이 없어짐

print(f'블록 {len(blocks)}개')                                   # → 1개 (본문과 표 구분 불가)

사내 공지 추석 연휴 근무 안내 연휴 기간 중 당직 근무자는 아래와 같습니다. 날짜 당직자 연락처 10/03 김민준 내선 2201 10/04 이서연 내선 2202
블록 1개


In [40]:
# [역할] HTML 좋은 예: script/style 제거 → 표를 먼저 떼어 두고 → 본문과 표를 각각 블록으로 만들기
# script, style 태그 묶어서 지우기
raw = re.sub('<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.S | re.I)   # 여는 태그~닫는 태그 사이까지 통째로 (re.S: 줄바꿈 포함, re.I: 대소문자 무시)

# 표 모아놓고 본문에서 동일한 패턴으로 태그 삭제
tables = re.findall('<table.*?</table>', raw, flags=re.S | re.I)   # 표 부분만 리스트로 모으기
body = re.sub('<table.*?</table>', ' ', raw, flags=re.S | re.I)    # 본문에서는 표를 빼기

# 남은 태그 지우고 공백 정리
body = re.sub('\\s+', ' ', re.sub('<[^>]+>', ' ', body)).strip()   # 태그 → 공백, 연속 공백 → 하나

# 떼어둔 표마다 태그를 지워 표 블럭으로 만들기
blocks = [('조항', page.name, body)] if body else []               # 본문이 있으면 첫 블록 (종류, 위치, 내용)
for i, table in enumerate(tables, 1):                              # 표마다 1번부터
    cleaned = re.sub('\\s+', ' ', re.sub('<[^>]+>', ' ', table)).strip()   # 표 안 태그 제거
    blocks.append(('표', f'{page.name} · 표{i}', cleaned))          # 표 블록 추가

# 카운트  
total = sum((len(t) for _, _, t in blocks))                        # 전체 글자 수
print(f'[좋은 예] 블록 {len(blocks)} · 표 {len(tables)} · 쪽 1 · 글자 {total}자')   # 요약
for n, (kind, locator, t) in enumerate(blocks, 1):                 # 블록마다
    print(f'  [{n}] {kind}  {locator:<26} {len(t):3d}자  {t[:40]}')   # 종류, 위치, 글자 수, 앞 40자

[좋은 예] 블록 2 · 표 1 · 쪽 1 · 글자 89자
  [1] 조항  sample_page.html            43자  사내 공지 추석 연휴 근무 안내 연휴 기간 중 당직 근무자는 아래와 같습
  [2] 표  sample_page.html · 표1       46자  날짜 당직자 연락처 10/03 김민준 내선 2201 10/04 이서연 내


HTML 안에는 &lt; &amp;와 같은 기능들을 

In [41]:
# [역할] 새로 만든 parser.parse()로 여러 형식을 한 번에 시험: txt/md/html은 성공, xlsx/pptx는 아직 미지원
import importlib                                      # 모듈 다시 불러오기용
from app.rag import parser                            # 형식별로 파서를 골라 주는 모듈
importlib.reload(parser)                              # parser.py 수정 내용 반영 (커널 재시작 없이)
targets = [FORMATS / 'sample_memo.txt', FORMATS / 'sample_guide.md', FORMATS / 'sample_page.html', FORMATS / 'sample_table.xlsx', FORMATS / 'sample_notice.pptx', SAMPLES / 'DOC-HR-014_국내출장_여비_규정_v2.0.docx']   # 시험할 파일들
for p in targets:                                     # 파일마다
    try:                                              # 실패해도 다음 파일로
        doc = parser.parse(p)                         # 확장자에 맞는 파서로 파싱
        print(f'{p.name:<40} 블록 {len(doc.blocks):3d}  표 {doc.table_count}  쪽 {doc.page_count}')   # 결과 요약
    except ValidationFailed as e:                     # 지원 안 하는 형식이면
        print(f'{p.name:<40} ValidationFailed: {e}')  # 오류 출력


sample_memo.txt                          블록   3  표 0  쪽 1
sample_guide.md                          블록   7  표 0  쪽 1
sample_page.html                         블록   2  표 1  쪽 1
sample_table.xlsx                        ValidationFailed: 지원하지 않는 형식입니다: .xlsx
sample_notice.pptx                       ValidationFailed: 지원하지 않는 형식입니다: .pptx
DOC-HR-014_국내출장_여비_규정_v2.0.docx          블록 233  표 7  쪽 1


In [42]:
# [역할] openpyxl로 엑셀 파일 열어 보기: 시트 목록과 각 행의 값을 출력
from openpyxl import load_workbook # workbook 엑셀 파일 worksheet 파일 안의 시트

XLSX=FORMATS / 'sample_table.xlsx'                  # 엑셀 파일 경로

wb=load_workbook(str(XLSX), data_only=True)         # 파일 열기 (data_only: 수식 대신 계산된 값)

print('시트 목록:', wb.sheetnames) # 시트 이름 목록

ws=wb[wb.sheetnames[0]] # 시트 이름으로 시트 한장 추출
print('시트 이름:', ws.title)                         # 고른 시트 이름

# 셀 객체 아닌 셀 안의 값들을 튜플로 행단위 추출
for row in ws.iter_rows(values_only=True):          # 한 행씩
    print(row)                                      # 값 튜플 출력

시트 목록: ['출장비 집계']
시트 이름: 출장비 집계
('부서', '1분기', '2분기', '3분기')
('인프라사업부 2팀', 4200000, 3800000, 5100000)
('구매팀', 1200000, 1400000, 1100000)
('보안팀', 900000, 1000000, 1300000)


In [43]:
# [역할] local_parsers에 추가한 parse_xlsx 확인: 시트가 마크다운 표 블록 하나로 바뀌는지 출력
import importlib                                                 # 모듈 다시 불러오기용
from app.rag import local_parsers                                # 로컬 파서 모듈
importlib.reload(local_parsers)                                  # 수정 내용 반영

doc = local_parsers.parse_xlsx(FORMATS / 'sample_table.xlsx')    # 엑셀 파싱
chars = sum((len(b.text) for b in doc.blocks))                   # 전체 글자 수
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count} · 글자 {chars}자')   # 요약
first = doc.blocks[0]                                            # 첫 블록
print('  kind    :', first.kind)                                 # 종류 (표)
print('  locator :', first.locator)                              # 위치 (시트 이름)
for line in first.text.split('\n')[:3]:                          # 마크다운 표 앞 3줄
    print(line)                                                  # 출력


블록 1 · 표 1 · 쪽 1 · 글자 161자
  kind    : 표
  locator : 출장비 집계
| 부서 | 1분기 | 2분기 | 3분기 |
|---|---|---|---|
| 인프라사업부 2팀 | 4200000 | 3800000 | 5100000 |


In [44]:
# [역할] 병합 셀이 있는 시험용 엑셀 파일 만들기 (A2:A3 병합)
from openpyxl import Workbook                        # 새 엑셀 파일 만들기용

wb = Workbook()                                      # 빈 통합문서
ws = wb.active                                       # 기본 시트

ws.title = '부서별 집행액'                              # 시트 이름
ws.append(['구분', '1분기', '2분기'])                   # 머리행
ws.append(['인프라사업부', 4200000, 3800000])           # 2행
ws.append(['인프라사업부', 1200000, 1400000])           # 3행 (A열은 병합되며 사라짐)
ws.append(['구매팀', 900000, 1000000])                  # 4행
ws.merge_cells('A2:A3')                              # A2~A3 병합 → A2 값만 남고 A3는 비게 됨
MERGED = SANDBOX / 'merged.xlsx'                     # 저장 경로
wb.save(str(MERGED))                                 # 저장

print(f'만든 파일 : {MERGED.relative_to(ROOT)} ({MERGED.stat().st_size:,} 바이트)')   # 경로와 크기


만든 파일 : sandbox\W5\Thu\merged.xlsx (5,022 바이트)


In [45]:
# [역할] 병합 셀 문제 확인: 병합된 아래 칸은 None으로 읽혀 표에서 부서 이름이 빠지는 것을 확인
wb=load_workbook(str(MERGED), data_only=True)        # 방금 만든 파일 열기
ws=wb.active                                         # 첫 시트

print('병합된 셀 범위:', [r.coord for r in ws.merged_cells.ranges])   # 병합 범위 목록

for row in ws.iter_rows(values_only=True):           # 한 행씩
    print(row)                                       # 3행 A열이 None

print()                                              # 빈 줄

print(local_parsers.parse_xlsx(MERGED).blocks[0].text)   # 파서 결과 표 출력
# 셀 병합된 부분은 None으로 마크다운에서는 내용 X => 예외 발생 X => TODO 병합 해결

병합된 셀 범위: ['A2:A3']
('구분', '1분기', '2분기')
('인프라사업부', 4200000, 3800000)
(None, 1200000, 1400000)
('구매팀', 900000, 1000000)

| 구분 | 1분기 | 2분기 |
|---|---|---|
| 인프라사업부 | 4200000 | 3800000 |
|  | 1200000 | 1400000 |
| 구매팀 | 900000 | 1000000 |


In [46]:
# [역할] python-pptx로 PPT 열어 보기: 모든 슬라이드의 텍스트 상자 글자를 모아 출력
from pptx import Presentation                        # PPT 파일 읽기용

PPTX=FORMATS / 'sample_notice.pptx'                  # PPT 파일 경로
prs=Presentation(str(PPTX)) # PPT 파일 열기

lines=[]                                             # 모은 텍스트
for slide in prs.slides:                             # 슬라이드마다
    for shape in slide.shapes:                       # 슬라이드 안 도형마다
        if shape.has_text_frame and shape.text.strip():   # 글자가 들어 있는 도형이면
            lines.append(shape.text.strip())         # 텍스트 모으기

body='\n'.join(lines)                                # 줄바꿈으로 이어 붙이기
print(f'슬라이드: {len(prs.slides)}장, {len(body)}자')   # 슬라이드 수, 글자 수
print(body)                                          # 내용 출력

슬라이드: 1장, 70자
출장 여비 규정 개정 안내
일비 25,000 → 30,000원
광역시 숙박비 상한 70,000원 신설
시행 2025-07-01


In [ ]:
# [역할] HANDLERS 등록 확인: xlsx·pptx가 분기표에 들어갔는지 보고, parse_local을 거쳐 실제로 파싱되는지 확인
print("HANDLERS 현재:", list(local_parsers.HANDLERS))       # 등록된 확장자 목록
print()                                                    # 빈 줄

# parse_xlsx 를 직접 부르지 않는다. parse_local 로 불러 '분기표를 거치는지' 를 본다.
for name in ("sample_table.xlsx", "sample_notice.pptx"):   # 새로 지원하는 두 형식
    doc = local_parsers.parse_local(FORMATS / name)        # 확장자 → HANDLERS에서 함수 찾아 파싱
    chars = sum(len(b.text) for b in doc.blocks)           # 전체 글자 수
    print(f"{name:<19} 블록 {len(doc.blocks)} - 표 {doc.table_count} - "   # 블록 수, 표 수
          f"쪽 {doc.page_count} - {chars:>3}자")                            # 쪽 수, 글자 수

In [ ]:
# [역할] 새 형식 5종을 parser.parse → chunk까지 통과시켜, 파싱 블록 수와 최종 청크 수를 비교
from app.rag import parser                         # 형식별로 파서를 골라 주는 모듈
from app.rag.chunker import chunk, summarize       # 청크 나누기, 청크 수 요약
importlib.reload(parser)                           # parser.py 수정 내용 반영
FIVE = ['sample_memo.txt', 'sample_guide.md', 'sample_page.html', 'sample_table.xlsx', 'sample_notice.pptx']   # 시험할 5개 파일
for name in FIVE:                                  # 파일마다
    doc = parser.parse(FORMATS / name)             # 파싱 → 블록
    chunks = chunk(doc)                            # 블록 → 청크 (조항은 합쳐지고, 표는 따로)
    print(f'{name:<19} 블록 {len(doc.blocks):>2}  표 {doc.table_count}  →  {summarize(chunks)}')   # 블록 수 → 청크 요약